# 02-SMOKE · End-to-end pipeline test on 1 game
Runs the **exact same stages as notebook 02** (guard -> clone -> seed -> extract -> SAVE -> verify) but on a **single game** (~3-4 min). If this completes and saves a dataset, Save & Run All on the full 25 is safe to launch unattended.

Saves to a throwaway dataset `vjepa-features-smoketest` (delete after).

**Settings:** GPU **T4**, Internet **ON**, attach **soccernet-25games**.

## Guard — T4 + videos attached

In [ ]:
import torch, glob
assert torch.cuda.is_available(), "No GPU. Set Accelerator = GPU T4 x2."
name = torch.cuda.get_device_name(0); print("GPU:", name)
assert "T4" in name, f"Got {name}; needs a T4 (P100 incompatible)."
vids = glob.glob("/kaggle/input/**/*_224p.mkv", recursive=True)
print("videos under /kaggle/input:", len(vids))
assert vids, "Attach soccernet-25games."


## Clone + install

In [ ]:
import os
%cd /kaggle/working
!rm -rf Football_JEPA
!git clone https://github.com/DanielSch02/Football_JEPA.git
%cd Football_JEPA
!git log --oneline -1
!pip install -q SoccerNet torchcodec decord


## Extract ONE game (both halves)

In [ ]:
from footy.config import VJEPA_GAME_PATHS_FULL, default_data_dir
import subprocess
VIDEO_DIR = default_data_dir()
WORK = "/kaggle/working/soccernet"
one = VJEPA_GAME_PATHS_FULL[2]   # game index 2 (Swansea-Man U): NOT one of the pre-done 3-game set's first two
print("VIDEO_DIR:", VIDEO_DIR)
print("test game:", one)
cmd = ["python", "-m", "scripts.extract_vjepa", "--data_dir", VIDEO_DIR,
       "--out_dir", WORK, "--batch", "8", "--games", one]
subprocess.run(cmd, check=True)


## Verify the feature file is real

In [ ]:
import glob, numpy as np
files = glob.glob(f"{WORK}/**/*_VJEPA21_L.npy", recursive=True)
print("feature files produced:", len(files))
assert files, "No features produced!"
for s in files:
    f = np.load(s)
    print(s.split("soccernet/")[-1], f.shape, "finite", bool(np.isfinite(f).all()), "std", round(float(f.std()),3))
    assert np.isfinite(f).all() and f.std() > 0, "bad features"
print("features look healthy")


## Test the SAVE mechanic (throwaway dataset)

In [ ]:
import json, subprocess
DATASET_ID = "daniels02/vjepa-features-smoketest"
json.dump({"title":"vjepa-features-smoketest","id":DATASET_ID,
           "licenses":[{"name":"CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json","w"))
r = subprocess.run(["kaggle","datasets","create","-p",WORK,"--dir-mode","tar"],
                   capture_output=True, text=True)
out = r.stdout + r.stderr
print(out)
if "already exists" in out.lower():
    r = subprocess.run(["kaggle","datasets","version","-p",WORK,"-m","smoke","--dir-mode","tar"],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
print("return code:", r.returncode)


## Result
If you see a dataset URL + `return code: 0` above, the **entire pipeline works**: extraction, feature quality, and save. -> Safe to run notebook 02 with **Save & Run All**.

Then delete the throwaway `vjepa-features-smoketest` dataset.